In [0]:
%sql
SELECT COUNT(*) FROM road_segments

In [0]:
%sql
WITH normalized AS (
    SELECT
        rs.segment_id,
        rs.corridor,
        rs.corridor_label,
        rs.midpoint_lon,
        rs.midpoint_lat,
        rs.straightness_score,
        tc.aadt,
        tc.ev_daily_vehicles,
        tc.avg_speed_mph,

        -- EV demand score
        LEAST(100.0, (tc.ev_daily_vehicles / 1200.0) * 100.0) AS ev_demand_score,

        -- Visibility score
        LEAST(100.0, (tc.aadt / 120000.0) * 100.0) AS visibility_score,

        -- Geometry score (already 0-100)
        rs.straightness_score AS geometry_score,

        -- Weather score (inverted)
        (100.0 - wr.winter_risk_score) AS weather_score,

        -- Safety score (inverted)
        GREATEST(0.0, 100.0 - (inc.annual_crashes_per_mile / 8.0) * 100.0) AS safety_score,

        -- Speed score (optimal 60-75 mph)
        CASE
            WHEN tc.avg_speed_mph BETWEEN 60 AND 75 THEN 100.0
            WHEN tc.avg_speed_mph < 60 THEN GREATEST(0.0, 100.0 - (60 - tc.avg_speed_mph) * 5.0)
            ELSE GREATEST(0.0, 100.0 - (tc.avg_speed_mph - 75) * 5.0)
        END AS speed_score

    FROM road_segments rs
    JOIN traffic_counts tc  ON tc.segment_id = rs.segment_id
    JOIN weather_risk   wr  ON wr.segment_id = rs.segment_id
    JOIN incidents      inc ON inc.segment_id = rs.segment_id
)

SELECT * FROM normalized LIMIT 5

In [0]:
%sql
WITH normalized AS (
    SELECT
        rs.segment_id,
        rs.corridor,
        rs.corridor_label,
        rs.midpoint_lon,
        rs.midpoint_lat,
        tc.aadt,
        tc.ev_daily_vehicles,
        tc.avg_speed_mph,
        rs.straightness_score,
        LEAST(100.0, (tc.ev_daily_vehicles / 1200.0) * 100.0) AS ev_demand_score,
        LEAST(100.0, (tc.aadt / 120000.0) * 100.0)            AS visibility_score,
        rs.straightness_score                                   AS geometry_score,
        (100.0 - wr.winter_risk_score)                         AS weather_score,
        GREATEST(0.0, 100.0 - (inc.annual_crashes_per_mile / 8.0) * 100.0) AS safety_score,
        CASE
            WHEN tc.avg_speed_mph BETWEEN 60 AND 75 THEN 100.0
            WHEN tc.avg_speed_mph < 60 THEN GREATEST(0.0, 100.0 - (60 - tc.avg_speed_mph) * 5.0)
            ELSE GREATEST(0.0, 100.0 - (tc.avg_speed_mph - 75) * 5.0)
        END AS speed_score
    FROM road_segments rs
    JOIN traffic_counts tc  ON tc.segment_id = rs.segment_id
    JOIN weather_risk   wr  ON wr.segment_id = rs.segment_id
    JOIN incidents      inc ON inc.segment_id = rs.segment_id
),

power_proximity AS (
    SELECT
        rs.segment_id,
        MIN(
            60.0 * SQRT(
                POWER((pi.lon - rs.midpoint_lon) * COS(RADIANS(rs.midpoint_lat)), 2)
                + POWER(pi.lat - rs.midpoint_lat, 2)
            )
        ) AS nearest_substation_miles
    FROM road_segments rs
    CROSS JOIN power_infra pi
    WHERE pi.type = 'substation'
    GROUP BY rs.segment_id
),

power_scored AS (
    SELECT
        segment_id,
        nearest_substation_miles,
        GREATEST(0.0, 100.0 - (nearest_substation_miles / 20.0) * 100.0) AS power_score
    FROM power_proximity
)

SELECT n.segment_id, n.corridor_label, 
       ROUND(pp.nearest_substation_miles, 1) AS miles_to_substation,
       ROUND(pp.power_score, 1) AS power_score
FROM normalized n
JOIN power_scored pp ON pp.segment_id = n.segment_id
LIMIT 5

In [0]:
%sql
WITH normalized AS (
    SELECT
        rs.segment_id,
        rs.corridor,
        rs.corridor_label,
        rs.midpoint_lon,
        rs.midpoint_lat,
        tc.aadt,
        tc.ev_daily_vehicles,
        tc.avg_speed_mph,
        rs.straightness_score,
        LEAST(100.0, (tc.ev_daily_vehicles / 1200.0) * 100.0) AS ev_demand_score,
        LEAST(100.0, (tc.aadt / 120000.0) * 100.0)            AS visibility_score,
        rs.straightness_score                                   AS geometry_score,
        (100.0 - wr.winter_risk_score)                         AS weather_score,
        GREATEST(0.0, 100.0 - (inc.annual_crashes_per_mile / 8.0) * 100.0) AS safety_score,
        CASE
            WHEN tc.avg_speed_mph BETWEEN 60 AND 75 THEN 100.0
            WHEN tc.avg_speed_mph < 60 THEN GREATEST(0.0, 100.0 - (60 - tc.avg_speed_mph) * 5.0)
            ELSE GREATEST(0.0, 100.0 - (tc.avg_speed_mph - 75) * 5.0)
        END AS speed_score
    FROM road_segments rs
    JOIN traffic_counts tc  ON tc.segment_id = rs.segment_id
    JOIN weather_risk   wr  ON wr.segment_id = rs.segment_id
    JOIN incidents      inc ON inc.segment_id = rs.segment_id
),

power_proximity AS (
    SELECT
        rs.segment_id,
        MIN(
            60.0 * SQRT(
                POWER((pi.lon - rs.midpoint_lon) * COS(RADIANS(rs.midpoint_lat)), 2)
                + POWER(pi.lat - rs.midpoint_lat, 2)
            )
        ) AS nearest_substation_miles
    FROM road_segments rs
    CROSS JOIN power_infra pi
    WHERE pi.type = 'substation'
    GROUP BY rs.segment_id
),

power_scored AS (
    SELECT
        segment_id,
        nearest_substation_miles,
        GREATEST(0.0, 100.0 - (nearest_substation_miles / 20.0) * 100.0) AS power_score
    FROM power_proximity
),

interchange_density AS (
    SELECT
        rs.segment_id,
        COUNT(ix.interchange_id) AS interchange_count
    FROM road_segments rs
    LEFT JOIN interchanges ix ON ix.segment_id = rs.segment_id
    GROUP BY rs.segment_id
),

interchange_scored AS (
    SELECT
        segment_id,
        interchange_count,
        GREATEST(0.0, 100.0 - interchange_count * 10.0) AS interchange_score
    FROM interchange_density
),

composite AS (
    SELECT
        n.segment_id,
        n.corridor,
        n.corridor_label,
        n.midpoint_lon,
        n.midpoint_lat,
        n.aadt,
        n.ev_daily_vehicles,
        ps.nearest_substation_miles,
        ixs.interchange_count,
        ROUND(n.ev_demand_score,    1) AS ev_demand_score,
        ROUND(n.visibility_score,   1) AS visibility_score,
        ROUND(n.geometry_score,     1) AS geometry_score,
        ROUND(n.weather_score,      1) AS weather_score,
        ROUND(n.safety_score,       1) AS safety_score,
        ROUND(n.speed_score,        1) AS speed_score,
        ROUND(ps.power_score,       1) AS power_score,
        ROUND(ixs.interchange_score,1) AS interchange_score,
        ROUND(
            n.ev_demand_score    * 0.20
            + n.visibility_score * 0.08
            + n.geometry_score   * 0.10
            + n.weather_score    * 0.10
            + n.safety_score     * 0.15
            + n.speed_score      * 0.07
            + ps.power_score     * 0.12
            + ixs.interchange_score * 0.10
        , 2) AS composite_score
    FROM normalized n
    JOIN power_scored       ps  ON ps.segment_id  = n.segment_id
    JOIN interchange_scored ixs ON ixs.segment_id = n.segment_id
)

SELECT *,
    RANK() OVER (ORDER BY composite_score DESC) AS overall_rank
FROM composite
ORDER BY composite_score DESC